### np.linalg.solve

In [3]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _deep_copy(mat):
    return [row[:] for row in mat]


def _gauss_elim_partial_pivot(A, b):
    """
    Solve A x = b using Gaussian elimination with partial pivoting.
    A: n x n list of lists
    b: list of n values
    Returns x: list of n values
    """
    n = len(A)
    # augment A with b → [A | b]
    aug = [A[i][:] + [b[i]] for i in range(n)]

    for col in range(n):

        # partial pivot: find row with max abs value in this column
        max_row = col
        max_val = abs(aug[col][col])
        for row in range(col + 1, n):
            if abs(aug[row][col]) > max_val:
                max_val = abs(aug[row][col])
                max_row = row

        # check for singular matrix
        if abs(aug[max_row][col]) < 1e-12:
            raise ValueError("Matrix is singular or nearly singular — no unique solution")

        # swap current row with max row
        aug[col], aug[max_row] = aug[max_row], aug[col]

        # eliminate below
        for row in range(col + 1, n):
            factor = aug[row][col] / aug[col][col]
            for j in range(col, n + 1):
                aug[row][j] -= factor * aug[col][j]

    # back substitution
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        x[i] = aug[i][n]
        for j in range(i + 1, n):
            x[i] -= aug[i][j] * x[j]
        x[i] /= aug[i][i]

    return x


def np_linalg_solve(a, b):
    """
    Pure-Python equivalent of np.linalg.solve.
    Solves a @ x = b for x.

    a: square matrix, shape (n, n)
    b: either:
       - 1-D list of length n → returns 1-D solution list
       - 2-D list of shape (n, k) → returns 2-D solution of shape (n, k)
    """
    shape_a = get_shape(a)
    shape_b = get_shape(b)

    if len(shape_a) != 2:
        raise ValueError("a must be a 2-D square matrix")

    if shape_a[0] != shape_a[1]:
        raise ValueError("a must be a square matrix")

    n = shape_a[0]

    # 1-D b: solve single system
    if len(shape_b) == 1:
        if shape_b[0] != n:
            raise ValueError(f"b must have length {n}, got {shape_b[0]}")
        return _gauss_elim_partial_pivot(a, b)

    # 2-D b: solve for each column of b separately
    if len(shape_b) == 2:
        if shape_b[0] != n:
            raise ValueError(f"b must have {n} rows, got {shape_b[0]}")
        k = shape_b[1]
        result_cols = []
        for j in range(k):
            b_col = [b[i][j] for i in range(n)]
            x_col = _gauss_elim_partial_pivot(a, b_col)
            result_cols.append(x_col)
        # transpose back to (n x k)
        return [[result_cols[j][i] for j in range(k)] for i in range(n)]

    raise ValueError("b must be 1-D or 2-D")

### test cases 

In [4]:
print(np_linalg_solve([[1, 2], [3, 5]], [1, 2]))
print(np_linalg_solve([[3, 1], [1, 2]], [9, 8]))
print(np_linalg_solve([[1, 0, 0], [0, 1, 0], [0, 0, 1]], [4, 5, 6]))
print(np_linalg_solve([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], [8, -11, -3]))
print(np_linalg_solve([[1, 2], [3, 5]], [[1, 0], [2, 1]]))
print(np_linalg_solve([[1, 0], [0, 1]], [3, 7]))

[-0.9999999999999994, 0.9999999999999997]
[2.0, 3.0]
[4.0, 5.0, 6.0]
[2.0, 3.0000000000000004, -0.9999999999999999]
[[-0.9999999999999994, 1.9999999999999993], [0.9999999999999997, -0.9999999999999996]]
[3.0, 7.0]


In [5]:
print(np_linalg_solve([[1, 2], [3]], [1, 2])) 

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)

In [6]:
print(np_linalg_solve([[1, 2, 3], [4, 5, 6]], [1, 2])) 

ValueError: a must be a square matrix

In [7]:
print(np_linalg_solve([[1, 2], [2, 4]], [1, 2])) 

ValueError: Matrix is singular or nearly singular — no unique solution

In [8]:
print(np_linalg_solve([[1, 2], [3, 4], [5, 6]], [1, 2, 3]))
 

ValueError: a must be a square matrix

In [9]:
print(np_linalg_solve([1, 2, 3], [1, 2, 3])) 

ValueError: a must be a 2-D square matrix